# Cibuco_Boriken — BirdCLEF+ 2026 Inference v7 (Perch Native)
**Team:** Cibuco_Boriken | **Model:** Google Perch (native classifier) | **234 species**

Uses Perch's **built-in species classifier** (trained on all of Xeno-Canto) instead of our custom MLP head.
Perch can predict 203/234 competition species natively.

Reference: https://www.kaggle.com/models/google/bird-vocalization-classifier

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q librosa audioread tensorflow
print('Dependencies installed ✅')

In [ ]:
# ── Cell 2: Imports + config ──
import os
import json
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# Paths
DATA_DIR = Path('/kaggle/input/competitions/birdclef-2026')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

# ⬇️ UPDATE THIS: path to the Perch model from Kaggle Models
# Attach: google/bird-vocalization-classifier → look for the SavedModel dir
PERCH_MODEL_DIR = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1')

# Config
SAMPLE_RATE = 32000
WINDOW_SEC  = 5.0

ZERO_SHOT_PRIOR = 1.0 / 234

print(f'Perch model dir: {PERCH_MODEL_DIR}')
print(f'Exists: {PERCH_MODEL_DIR.exists()}')
print('Config ready ✅')

In [ ]:
# ── Cell 3: Load Perch v2 + smoke test ──
perch = tf.saved_model.load(str(PERCH_MODEL_DIR))
infer_fn = perch.signatures['serving_default']

# Quick smoke test: 5s of silence → should return label logits
test_wav = tf.constant(np.zeros((1, int(SAMPLE_RATE * WINDOW_SEC)), dtype=np.float32))
test_out = infer_fn(inputs=test_wav)
assert 'label' in test_out, f"Expected 'label' key, got: {list(test_out.keys())}"
num_perch_classes = test_out['label'].shape[-1]
print(f'Perch v2 loaded ✅  |  output classes: {num_perch_classes}  |  keys: {list(test_out.keys())}')

In [ ]:
# ── Cell 4: Build species mapping ──
taxonomy_df = pd.read_csv(DATA_DIR / 'taxonomy.csv')
SPECIES = taxonomy_df['primary_label'].tolist()
NUM_SPECIES = len(SPECIES)

# Load Perch label index — try labels.csv, fall back to ebird_classes csv
label_file = PERCH_MODEL_DIR / 'assets' / 'labels.csv'
if not label_file.exists():
    # Fall back: search assets dir for any CSV
    assets_dir = PERCH_MODEL_DIR / 'assets'
    csvs = sorted(assets_dir.glob('*.csv')) if assets_dir.exists() else []
    assert len(csvs) > 0, f'No label CSV found in {assets_dir}'
    label_file = csvs[0]
    print(f'Using fallback label file: {label_file.name}')

label_df = pd.read_csv(label_file)
print(f'Label file: {label_file.name} | shape: {label_df.shape} | cols: {label_df.columns.tolist()[:5]}')

# Detect the eBird code column
ebird_col = label_df.columns[0]
for col in ['ebird2021', 'ebird_code', 'species_code']:
    if col in label_df.columns:
        ebird_col = col
        break

perch_labels = label_df[ebird_col].astype(str).tolist()
perch_to_idx = {sp: i for i, sp in enumerate(perch_labels)}

mapped = [sp for sp in SPECIES if sp in perch_to_idx]
unmapped = [sp for sp in SPECIES if sp not in perch_to_idx]
print(f'Perch species: {len(perch_labels)} | Mapped: {len(mapped)}/{NUM_SPECIES} | Unmapped: {len(unmapped)}')
if unmapped:
    print(f'Zero-shot species: {unmapped[:10]}{"..." if len(unmapped) > 10 else ""}')

In [ ]:
# ── Cell 5: Parse sample submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'Sample submission: {len(sample_sub)} rows, {len(sample_sub.columns)} cols')

def parse_row_id(row_id):
    parts = row_id.rsplit('_', 1)
    return parts[0], int(parts[1])

file_windows = defaultdict(list)
for row_id in sample_sub['row_id']:
    stem, end_time = parse_row_id(row_id)
    file_windows[stem].append((row_id, end_time))

print(f'Soundscape files: {len(file_windows)}')
print('Row IDs parsed ✅')

In [ ]:
# ── Cell 6: Perch native inference ──
import time

soundscape_dir = DATA_DIR / 'test_soundscapes'
window_samples = int(WINDOW_SEC * SAMPLE_RATE)

all_rows = {}
t_start = time.time()
total_windows = sum(len(v) for v in file_windows.values())
processed = 0

for file_stem in tqdm(sorted(file_windows.keys()), desc='Perch Native'):
    # Find audio file
    audio_path = None
    for ext in ['.ogg', '.wav', '.flac']:
        candidate = soundscape_dir / f'{file_stem}{ext}'
        if candidate.exists():
            audio_path = candidate
            break

    if audio_path is None:
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    try:
        audio, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'Error loading {audio_path}: {e}')
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    for row_id, end_time in file_windows[file_stem]:
        start_sample = max(0, int((end_time - WINDOW_SEC) * SAMPLE_RATE))
        end_sample = int(end_time * SAMPLE_RATE)
        chunk = audio[start_sample:end_sample]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))

        # Run Perch via serving_default signature
        waveform = tf.constant(chunk[np.newaxis, :], dtype=tf.float32)
        output = infer_fn(inputs=waveform)

        # 'label' output has shape (1, 14795) — raw logits for all Perch species
        logits = output['label'].numpy().squeeze()  # (14795,)

        # Convert logits to probabilities
        probs = 1.0 / (1.0 + np.exp(-logits))  # sigmoid

        # Build row using species mapping
        row = {'row_id': row_id}
        for sp in SPECIES:
            if sp in perch_to_idx:
                row[sp] = float(probs[perch_to_idx[sp]])
            else:
                row[sp] = ZERO_SHOT_PRIOR
        all_rows[row_id] = row

        processed += 1
        if processed % 100 == 0:
            elapsed = time.time() - t_start
            rate = processed / elapsed
            remaining = (total_windows - processed) / rate / 60
            print(f'  {processed}/{total_windows} windows | {elapsed/60:.1f} min elapsed | ~{remaining:.1f} min remaining')

elapsed = time.time() - t_start
print(f'\nTotal rows: {len(all_rows)} (expected: {len(sample_sub)})')
print(f'Total time: {elapsed/60:.1f} min ✅')

In [ ]:
# ── Cell 7: Generate submission.csv ──
rows_ordered = [all_rows[rid] for rid in sample_sub['row_id']]
submission_df = pd.DataFrame(rows_ordered)
submission_df = submission_df[sample_sub.columns]
submission_df = submission_df.fillna(ZERO_SHOT_PRIOR)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Submission saved ✅')
print(f'Shape: {submission_df.shape} (expected: {sample_sub.shape})')
print(f'Min prob: {submission_df.iloc[:, 1:].min().min():.6f}')
print(f'Max prob: {submission_df.iloc[:, 1:].max().max():.6f}')
print(submission_df.head(3))

In [ ]:
# ── Cell 8: Validate submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
our_sub    = pd.read_csv(OUTPUT_PATH)

print('=== Submission Validation ===')
print(f'Expected rows:  {len(sample_sub)}')
print(f'Our rows:       {len(our_sub)}')
print(f'Expected cols:  {len(sample_sub.columns)}')
print(f'Our cols:       {len(our_sub.columns)}')
print(f'Columns match:  {set(sample_sub.columns) == set(our_sub.columns)}')
print(f'Row IDs match:  {set(sample_sub["row_id"]) == set(our_sub["row_id"])}')
print(f'No NaN:         {our_sub.isna().sum().sum() == 0}')
print(f'No zeros:       {(our_sub.iloc[:, 1:] == 0).sum().sum() == 0}')
print()
if set(sample_sub.columns) == set(our_sub.columns):
    print('✅ PERCH NATIVE SUBMISSION VALID — ready to submit!')
else:
    missing = set(sample_sub.columns) - set(our_sub.columns)
    extra   = set(our_sub.columns) - set(sample_sub.columns)
    if missing: print(f'⚠️  Missing columns: {missing}')
    if extra:   print(f'⚠️  Extra columns: {extra}')